# How bootstrap aggregation (Bagging) works

That's a great question. The key idea is that **bagging (Bootstrap Aggregation)** is *not* trying to create better training sets individually. Instead, it's trying to create **different models** whose errors cancel each other out.

Here's how it works.

Suppose your original dataset has **100 samples**.

For each bootstrap sample:

* You randomly pick 100 samples **with replacement**.
* Since sampling is with replacement:

  * Some examples appear multiple times.
  * Some examples don't appear at all.

A typical bootstrap sample might look like:

```
Original:
A B C D E F G H I J

Bootstrap sample:
A C C D F G G G I J
```

Notice:

* `C` appears twice.
* `G` appears three times.
* `B`, `E`, and `H` are missing.

So each model sees a **slightly different version of the data**.

### Why do duplicates help?

The duplicates themselves aren't what improve accuracy.

They simply create randomness.

Imagine training three decision trees.

Tree 1:

```
A B C D E F G H I J
```

Tree 2:

```
A C C D F G G G I J
```

Tree 3:

```
B B C D E F H I I J
```

Since each tree gets a different dataset, each tree chooses different splits.

For example:

* Tree 1 might split on Age first.
* Tree 2 might split on Income first.
* Tree 3 might split on Education first.

Each tree makes different mistakes.

### Averaging reduces variance

Suppose you're predicting house prices.

```
Tree 1:  $210k
Tree 2:  $195k
Tree 3:  $205k
```

Average:

```
(210 + 195 + 205) / 3 = $203k
```

The average prediction is usually more stable than relying on a single tree.

For classification, the trees vote.

```
Tree 1 → Cat
Tree 2 → Dog
Tree 3 → Cat
Tree 4 → Cat
Tree 5 → Dog

Majority vote → Cat
```

Even if a few trees make mistakes, the majority often gets the correct answer.

### But aren't we losing information because some samples are omitted?

Yes, each bootstrap sample leaves out some training examples.

In fact, for a dataset of size (n), a bootstrap sample contains about **63.2% unique examples** on average. The remaining positions are filled by duplicates.

This seems wasteful, but it's actually beneficial because:

* Every tree sees a different subset of the data.
* The trees become less correlated.
* Averaging less-correlated models reduces variance.

### Intuition

Think of asking 100 students to solve the same math problem.

If every student studies from exactly the same notes, they'll tend to make the same mistake.

If each student studies from a slightly different set of examples, they'll make different mistakes.

Taking the majority answer is more reliable than trusting any one student.

### Why is this especially effective for decision trees?

Decision trees are **high-variance** models:

* Small changes in the training data can produce very different trees.

Bagging takes advantage of this. By training many trees on different bootstrap samples and combining their predictions, it greatly reduces variance without increasing bias much. That's why algorithms like Random Forest, which combine bagging with random feature selection, often perform much better than a single decision tree.

So, the duplicates are not the source of the improvement. They are a side effect of sampling with replacement, which creates diverse training sets. The **real improvement comes from averaging many diverse models**, which reduces variance and makes predictions more robust.

# -------------------------------------------------

# so we make the model predict with the sample datas and take the mean of it?

Exactly—but there's one important distinction.

We **don't average the predictions on the training (sample) data**. We average the predictions for **new, unseen data**.

Here's the process:

### Step 1: Create bootstrap datasets

Suppose your original dataset is:

```
A B C D E
```

Create three bootstrap samples:

```
Model 1: A B C C E
Model 2: A A D E E
Model 3: B C D D E
```

### Step 2: Train separate models

```
Bootstrap Sample 1 → Train Model 1
Bootstrap Sample 2 → Train Model 2
Bootstrap Sample 3 → Train Model 3
```

Now you have three different models.

### Step 3: Predict on a new data point

Suppose a new customer arrives whose label is unknown.

Each model predicts independently:

```
Model 1 → 85
Model 2 → 92
Model 3 → 89
```

For **regression**, the final prediction is:

[
\frac{85+92+89}{3}=88.67
]

For **classification**, suppose the predictions are:

```
Model 1 → Spam
Model 2 → Not Spam
Model 3 → Spam
```

The final prediction is **Spam** (majority vote).

---

### Why does this improve accuracy?

Each individual model is a little "wrong" in a different way because it was trained on a different bootstrap sample.

For example:

```
True value = 90

Model 1 predicts 85  (−5)
Model 2 predicts 92  (+2)
Model 3 predicts 89  (−1)

Average = 88.67
```

The errors tend to cancel each other out, making the ensemble prediction closer to the true value than many individual models.

This is why bagging is especially effective with **unstable models** like decision trees: a small change in the training data can produce a different tree, and averaging many such trees produces a much more stable and accurate predictor.

So your understanding is almost correct—the only adjustment is:

* ❌ Average predictions on the training samples.
* ✅ Train on different bootstrap samples, then average (or vote on) the predictions for **new/unseen data**.


## what does  boundaries = np.percentile(stat, [100 * alpha / 2.0, 100 * (1 - alpha / 2.0)])

Great question! Let's use a **small concrete example**.

Suppose you have:

```python
stat = [5.2, 5.6, 6.0, 6.1, 5.9, 5.8, 6.2, 5.7]
alpha = 0.05
```

The function computes:

```python
boundaries = np.percentile(stat, [2.5, 97.5])
```

### What does `np.percentile()` return?

It returns **two numbers**:

1. The value at the **2.5th percentile** (lower bound).
2. The value at the **97.5th percentile** (upper bound).

For this example, it might return something like:

```python
boundaries = [5.27, 6.18]
```

So `boundaries` is simply a NumPy array containing the lower and upper limits:

```python
array([5.27, 6.18])
```

When the function executes:

```python
return boundaries
```

it returns:

```python
array([5.27, 6.18])
```

---

### Another simple example

Suppose

```python
stat = [1, 2, 3, 4, 5]
alpha = 0.20
```

Then:

```python
100 * alpha / 2 = 10
100 * (1 - alpha / 2) = 90
```

So the function computes:

```python
np.percentile(stat, [10, 90])
```

which returns approximately:

```python
array([1.4, 4.6])
```

Thus:

```python
boundaries = array([1.4, 4.6])
```

---

### So what exactly is stored in `boundaries`?

It always has **two values**:

```python
boundaries = [lower_bound, upper_bound]
```

For example:

```python
boundaries = [4.82, 5.31]
```

where:

* `boundaries[0]` → lower confidence limit
* `boundaries[1]` → upper confidence limit

You could even do:

```python
lower, upper = stat_intervals(stat, 0.05)

print(lower)
print(upper)
```

Output:

```python
4.82
5.31
```

So, **`boundaries` is just a two-element array containing the confidence interval: `[lower_bound, upper_bound]`.**


## wont it return the values from stats array

Excellent observation! This is one of the most common questions about `np.percentile()`.

The answer is:

> **Not always.** `np.percentile()` can return values **between** the values in `stat` because it **interpolates** when the requested percentile doesn't fall exactly on one of the data points.

Let's see why.

---

### Suppose you have

```python
stat = [1, 2, 3, 4, 5]
```

There are only **5 values**.

Now ask for the **50th percentile**:

```python
np.percentile(stat, 50)
```

This returns

```python
3
```

because 3 is the middle value.

---

### But what about the 10th percentile?

You ask:

```python
np.percentile(stat, 10)
```

The 10th percentile is **not exactly at any one of the five values**.

It lies somewhere between the first and second values.

NumPy estimates that position and returns approximately

```python
1.4
```

Notice:

```
1 -------- 2
    ↑
   1.4
```

So **1.4 isn't in the original array**, but it represents where the 10th percentile falls.

---

### Another example

```python
stat = [10, 20, 30, 40]
```

If you ask for the 25th percentile:

```python
np.percentile(stat, 25)
```

you don't necessarily get

```python
20
```

Instead, NumPy computes a position between 10 and 20 (depending on its interpolation method) and returns a value such as

```python
17.5
```

Again, **17.5 isn't one of the original values**.

---

### Why does NumPy do this?

Percentiles represent **positions in a distribution**, not necessarily actual observations.

When the desired position falls between two data points, NumPy interpolates to estimate the percentile value.

---

### Can it return an actual value from `stat`?

Yes! If the requested percentile lands exactly on one of the data points.

For example:

```python
stat = [1, 2, 3, 4, 5]

np.percentile(stat, 50)
```

returns

```python
3
```

because the median is exactly the third value.

---

### In the context of your function

```python
boundaries = np.percentile(stat, [2.5, 97.5])
```

If `stat` contains thousands of bootstrap estimates, such as

```python
stat = [5.01, 5.03, 5.04, ..., 5.98, 6.00]
```

then the 2.5th and 97.5th percentiles will often fall **between** two neighboring values. That's why `boundaries` may contain numbers that weren't originally in `stat`.

So the returned interval is an **estimated boundary**, not necessarily two values copied directly from the array.


# ------------------------

# Bootstrapping code explination

This is a classic example of **using bootstrap to estimate a confidence interval for the mean**. Let's go through it one line at a time.

---

## Step 1: The bootstrap function

```python
def get_bootstrap_samples(data, n_samples):
    """Generate bootstrap samples using the bootstrap method."""
    indices = np.random.randint(0, len(data), (n_samples, len(data)))
    samples = data[indices]
    return samples
```

### Suppose

```python
data = np.array([10, 20, 30, 40, 50])
```

This is your original sample.

```
Index : 0   1   2   3   4
Value :10  20  30  40  50
```

Assume

```python
n_samples = 3
```

---

### Line 1

```python
indices = np.random.randint(0, len(data), (n_samples, len(data)))
```

`len(data)` is 5.

So this becomes

```python
indices = np.random.randint(0, 5, (3,5))
```

Suppose NumPy generates

```python
indices =
[[2, 4, 1, 0, 2],
 [1, 1, 3, 4, 0],
 [4, 3, 2, 2, 1]]
```

Notice:

* 3 rows → because we asked for 3 bootstrap samples
* 5 columns → because each bootstrap sample has the same size as the original data

---

### Line 2

```python
samples = data[indices]
```

NumPy replaces every index with the corresponding value.

```
data =
[10,20,30,40,50]
```

So

```
2 → 30
4 → 50
1 → 20
0 → 10
2 → 30
```

The first row becomes

```
[30,50,20,10,30]
```

The whole result becomes

```python
samples =
[[30,50,20,10,30],
 [20,20,40,50,10],
 [50,40,30,30,20]]
```

Each row is one bootstrap sample.

Then

```python
return samples
```

returns this matrix.

---

# Step 2: Confidence interval function

```python
def stat_intervals(stat, alpha):
```

Here

```
stat
```

is **not the original data**.

It is a list of statistics.

For example

```python
stat =
[27.5,
28.1,
29.6,
30.2,
26.8,
...]
```

Each number is the **mean of one bootstrap sample**.

---

### This line

```python
boundaries = np.percentile(
    stat,
    [100*alpha/2,
     100*(1-alpha/2)]
)
```

Suppose

```
alpha = 0.05
```

Then

```
100*0.05/2 = 2.5

100*(1-0.05/2)=97.5
```

So it becomes

```python
np.percentile(stat,[2.5,97.5])
```

This returns

```
[27.3,30.8]
```

meaning

> "95% of the bootstrap means lie between 27.3 and 30.8."

That becomes the bootstrap confidence interval.

---

# Step 3: Separate the data

```python
loyal_calls = telecom_data.loc[
    telecom_data["Churn"] == False,
    "Customer service calls"
].values
```

Suppose the dataframe is

| Customer | Churn | Calls |
| -------- | ----- | ----- |
| A        | False | 1     |
| B        | True  | 5     |
| C        | False | 2     |
| D        | False | 1     |
| E        | True  | 6     |

This line extracts only loyal customers.

```
loyal_calls

[1,2,1]
```

Similarly

```python
churn_calls
```

becomes

```
[5,6]
```

---

# Step 4

```python
np.random.seed(0)
```

This fixes the random number generator.

Without it

every run gives different bootstrap samples.

With it

every run produces exactly the same samples.

Useful for reproducible experiments.

---

# Step 5

```python
loyal_mean_scores = [
    np.mean(sample)
    for sample in get_bootstrap_samples(loyal_calls,1000)
]
```

Let's expand it.

First,

```python
get_bootstrap_samples(loyal_calls,3)
```

might return

```python
[[1,2,1],
 [2,2,1],
 [1,1,1]]
```

Now the loop does

First sample

```
[1,2,1]

mean = 1.33
```

Second

```
[2,2,1]

mean = 1.67
```

Third

```
[1,1,1]

mean = 1
```

So

```python
loyal_mean_scores

[1.33,
1.67,
1]
```

In the actual code

```
1000
```

samples are generated.

So you'll have

```
1000 bootstrap means
```

like

```
[
1.33,
1.67,
1.00,
1.67,
1.33,
...
]
```

---

Exactly the same thing happens for churn customers.

---

# Step 6

```python
stat_intervals(loyal_mean_scores,0.05)
```

Suppose

```
1000 bootstrap means

[
1.1,
1.2,
1.3,
...
2.0
]
```

The function finds

```
2.5th percentile = 1.18

97.5th percentile = 1.82
```

So it prints

```
Service calls from loyal:
mean interval
[1.18,1.82]
```

---

# The entire workflow

```text
Original data
      │
      ▼
Generate 1000 bootstrap samples
      │
      ▼
Compute mean of each sample
      │
      ▼
1000 bootstrap means
      │
      ▼
Take 2.5th and 97.5th percentiles
      │
      ▼
95% confidence interval for the true mean
```

### Why do all this?

You only have **one observed sample** of customers, but you want to estimate how uncertain the sample mean is. Bootstrapping simulates the process of repeatedly sampling from the population by repeatedly resampling **from your observed sample with replacement**. The distribution of those 1000 bootstrap means approximates the sampling distribution of the mean, allowing you to estimate a confidence interval without assuming a particular underlying distribution.


# --------------------------------------------------------

# Out of the bag error

Absolutely! Let's start from the beginning with a very simple example.

---

## Step 1: Imagine you have 10 students

Suppose your dataset is:

```text
A B C D E F G H I J
```

These are the 10 training examples.

---

## Step 2: Build the first tree

To build **Tree 1**, Random Forest does **not** use all 10 examples directly.

Instead, it randomly picks **10 examples with replacement**.

It might pick:

```text
A C C E F F H I I J
```

Notice:

* **C** is picked twice.
* **F** is picked twice.
* **I** is picked twice.
* **B, D, and G** are never picked.

So:

**Used for Tree 1:**

```text
A C E F H I J
```

**Left out for Tree 1:**

```text
B D G
```

These left-out examples are **not used to build Tree 1**.

---

## Step 3: Build the second tree

Now Random Forest builds **Tree 2**.

It **doesn't reuse the same sample**.

It creates a **new random sample**:

```text
B D E G G H J A B C
```

Now:

**Used for Tree 2:**

```text
A B C D E G H J
```

**Left out for Tree 2:**

```text
F I
```

Notice something?

* In Tree 1, **B** was left out.
* In Tree 2, **B** is used.

So being **left out** only applies to **one tree**.

---

## Step 4: What does "with replacement" mean?

Suppose you pick **C**.

Instead of removing it, you **put it back** before the next pick.

So you can pick **C** again.

That's why the sample can look like:

```text
A C C E F F H I I J
```

Some examples appear multiple times.

Some never appear.

---

## Step 5: Why 37%?

Because of this random sampling with replacement:

* Some examples are picked multiple times.
* Some are never picked.

For a large dataset, on average:

* **63%** of the examples appear at least once.
* **37%** never appear.

So if there are **1000 examples**:

* About **630** are used to build the tree.
* About **370** are left out.

---

## Step 6: What does the book mean?

When it says:

> "Approximately 37% of inputs are left out of a particular bootstrap sample and are not used in the construction of the k-th tree."

It simply means:

> "When building **one tree**, around 37% of the training examples are not selected, so they are **not used for that tree**."

It **does not** mean they are ignored forever.

The next tree gets a **new random sample**, so those examples will probably be used there.

---

### Think of it like this 🎲

Imagine you have **100 flashcards**.

To teach **Teacher 1**, you randomly choose 100 cards **with replacement**.

Some cards are repeated, so about **37 cards are never shown** to Teacher 1.

Now for **Teacher 2**, you shuffle and repeat the process.

This time, a different set of about 37 cards is left out.

After training many teachers (trees), **every flashcard has been seen by many teachers**, just not by every single one.

This is exactly how a **Random Forest** works:

* 🌳 Each tree learns from a **different random subset** of the data.
* 🌳 The trees make different mistakes.
* 🌳 Combining all their predictions produces a stronger, more accurate model.


# ---------------------------------------------------

# Image and the explination

Of course! Let's completely ignore the complicated words and pretend we're teaching this to someone who has never heard of Random Forests.

---

# Step 1: Look at the top picture

The **top picture** is **all of your data**.

Imagine these are fruits.

* 🟢 = Apples
* 🔴 = Oranges

So this is your whole dataset.

```
Whole Dataset
🟢 🔴 🟢 🔴 🟢 ...
```

---

# Step 2: Build one decision tree

To build **one tree**, we **do not use every point**.

We randomly pick some of them.

So suppose we picked these:

```
✓ ✓ ✓ ✓ ✓ ✓
```

and didn't pick these:

```
✗ ✗ ✗ ✗
```

The ✓ points become the **training data**.

This is the **bottom-left picture**.

**Question:**

> Why don't we use all the data?

Because Random Forest wants **every tree to be a little different**.

---

# Step 3: The tree learns

Using only the **bottom-left points**, the decision tree learns how to separate red and green.

The grid represents what the tree learned.

For example, it learns rules like:

```
If point is here → Green

If point is here → Red
```

The tree has **never seen** the ✗ points.

---

# Step 4: Now look at the bottom-right picture

These are exactly the points that were **left out**.

The tree has never seen them before.

Now we ask the tree:

> "Can you guess the color of these points?"

The tree starts predicting.

---

Suppose there are 15 points.

The tree predicts:

| Point | Actual | Prediction |
| ----- | ------ | ---------- |
| 1     | 🟢     | 🟢 ✅       |
| 2     | 🔴     | 🔴 ✅       |
| 3     | 🟢     | 🟢 ✅       |
| 4     | 🔴     | 🟢 ❌       |
| 5     | 🟢     | 🟢 ✅       |
| ...   | ...    | ...        |

Most are correct.

Some are wrong.

---

# Step 5: Orange boxes

The orange boxes simply say:

> **"The tree made a mistake here."**

Nothing more.

For example,

Actual:

```
🟢
```

Tree says:

```
🔴
```

Wrong.

That point gets highlighted.

There are **4 mistakes**.

---

# Step 6: Why 11/15?

There are

```
15 points to test
```

The tree gets

```
11 correct
```

and

```
4 wrong
```

So

```
Accuracy =
Correct
--------
Total

=

11
--
15

= 73.33%
```

---

# The whole image in one story

Imagine you're a teacher.

You have **30 students**.

You decide:

> "I'll teach only 20 students today."

Those 20 students are the **bottom-left picture**.

The other 10 students stay outside.

After teaching, you call those 10 students in and give them a quiz.

If they answer well,

you know your teaching worked.

If they answer poorly,

you know your teaching wasn't very good.

Those 10 students are the **bottom-right picture**.

---

# This is exactly what Random Forest does

For **Tree 1**

```
Teach using 63% of data
↓

Test on the other 37%
```

For **Tree 2**

```
Choose a different 63%
↓

Test on a different 37%
```

For **Tree 3**

```
Choose another different 63%
↓

Test on another different 37%
```

Every tree gets:

* its **own training data**, and
* its **own small test set (OOB samples)**.

---

## The one sentence to remember

**The left picture is the data used to teach one tree. The right picture is the data that was not used to teach that tree, so it is used to test how well that tree learned.**


# -----------------------------------------------------

# points explination - Out-of-Bag error is computed in the following way

Yes! This is the **last piece** of the Random Forest puzzle. Let's make it very simple with an example.

---

# Suppose we have 3 trees

And our dataset has 5 data points.

```text
A  B  C  D  E
```

When Random Forest builds the trees, each tree uses a different bootstrap sample.

Let's say:

| Tree   | Used for training | Left out (OOB) |
| ------ | ----------------- | -------------- |
| Tree 1 | A B C             | D E            |
| Tree 2 | A D E             | B C            |
| Tree 3 | B C D             | A E            |

Notice something?

Every data point is **left out for some trees**.

For example:

* A is OOB for Tree 3.
* B is OOB for Tree 2.
* E is OOB for Trees 1 and 3.

---

## Point 1

> **Take all instances that have been chosen as a part of test set for some tree.**

This simply means:

> Collect every data point that was left out of at least one tree.

In our example:

```text
A B C D E
```

Every point was left out somewhere.

These become the **OOB dataset**.

---

## Point 2

> **Take one instance from the OOB dataset and all models (trees) that were not trained with this instance.**

Let's choose **Point E**.

Which trees never saw E?

| Tree   | Saw E? |
| ------ | ------ |
| Tree 1 | ❌ No   |
| Tree 2 | ✅ Yes  |
| Tree 3 | ❌ No   |

So only

* Tree 1
* Tree 3

can vote on E.

Why?

Because Tree 2 already learned from E.

Testing Tree 2 on E would be cheating.

---

## Point 3

> **Compare the majority vote of these trees with the true label.**

Suppose E is actually:

```text
Green
```

The predictions are

| Tree   | Prediction |
| ------ | ---------- |
| Tree 1 | Green      |
| Tree 3 | Red        |

Now we vote.

Green → 1 vote

Red → 1 vote

Suppose there were another tree:

| Tree 5 | Green |

Now the votes become

Green → 2

Red → 1

Majority vote = Green

Actual label = Green

✅ Correct prediction.

---

Another example.

Suppose point B is actually

```text
Red
```

Only Tree 2 never saw B.

Tree 2 predicts

```text
Green
```

Majority vote (only one vote)

```text
Green
```

Actual

```text
Red
```

❌ Wrong.

---

## Point 4

> **Do this for every OOB instance.**

Repeat exactly the same thing.

For A

↓

Find trees that never saw A

↓

Let them vote

↓

Check if correct

---

For B

↓

Find trees that never saw B

↓

Vote

↓

Check

---

For C

↓

Repeat

---

Eventually you'll have something like

| Data Point | Correct? |
| ---------- | -------- |
| A          | ✅        |
| B          | ❌        |
| C          | ✅        |
| D          | ✅        |
| E          | ✅        |

---

Suppose

4 are correct

1 is wrong

Then

Accuracy

```text
4/5 = 80%
```

OOB Error

```text
1/5 = 20%
```

---

# The entire process in one picture

```text
Original data

A  B  C  D  E
```

Build Tree 1

```text
Uses:
A B C

Tests:
D E
```

Build Tree 2

```text
Uses:
A D E

Tests:
B C
```

Build Tree 3

```text
Uses:
B C D

Tests:
A E
```

Now evaluate:

```text
A
↓
Only Tree 3 predicts
↓
Correct?
```

```text
B
↓
Only Tree 2 predicts
↓
Correct?
```

```text
E
↓
Tree 1 and Tree 3 predict
↓
Take majority vote
↓
Correct?
```

Repeat for every point.

---

## The key idea to remember ⭐

The clever trick behind **Out-of-Bag (OOB)** is:

* **A tree is never tested on the data it was trained with.**
* For each training example, Random Forest **only uses the trees that never saw that example** to make a prediction.
* Comparing those predictions with the true labels gives an estimate of the model's performance **without needing a separate validation set**.


# -----------------------------------------------------

Not quite. The **out-of-bag (OOB) error** is **not** calculated by comparing the in-bag predictions to the out-of-bag predictions. Instead, it compares the **OOB prediction to the true label**.

Here's how it works:

### Step 1: Bootstrap sampling

Suppose you have 100 training samples.

For each tree in a random forest:

* Sample 100 observations **with replacement**.
* About **63.2%** of the unique observations end up in the bootstrap sample (the "in-bag" data).
* About **36.8%** are **not selected**—these are the **out-of-bag (OOB)** samples for that tree.

### Step 2: Train the tree

The tree is trained **only on the in-bag samples**.

### Step 3: Predict OOB samples

For every observation that was **not used** to train that tree:

* Pass it through the tree.
* Record the prediction.

### Step 4: Aggregate OOB predictions

Each observation is OOB for many trees (roughly one-third of all trees).

For a particular observation:

* Collect predictions from **only the trees where it was OOB**.
* Aggregate them:

  * **Classification:** majority vote
  * **Regression:** average prediction

### Step 5: Compute OOB error

Compare the aggregated OOB prediction with the **actual target value**.

For classification:

[
\text{OOB Error}
================

\frac{\text{Number of incorrectly classified observations}}{\text{Total number of observations}}
]

For regression:

Use metrics such as

[
\text{MSE}
==========

\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i^{OOB})^2
]

where (\hat y_i^{OOB}) is the prediction obtained only from trees for which observation (i) was out-of-bag.

---

### Example

Suppose observation #5 has the true label:

| True label | A |
| ---------- | - |

It is OOB for Trees 2, 4, 7, and 10.

Their predictions are:

| Tree | Prediction |
| ---- | ---------- |
| 2    | A          |
| 4    | B          |
| 7    | A          |
| 10   | A          |

Majority vote = **A**

Compare:

* OOB prediction = A
* True label = A

Correct prediction → contributes **0** to the OOB error.

---

### What the in-bag predictions are used for

The in-bag samples are **only used to train each tree**. Their predictions are **not** used in the OOB error calculation because those predictions would be optimistically biased—the tree has already seen those samples during training.

---

So, the key idea is:

* ❌ **Not:** compare in-bag prediction vs. OOB prediction.
* ✅ **Instead:** compare the **aggregated OOB prediction** (from trees that did not see the observation) with the **true label**. This makes the OOB error a built-in estimate of the model's generalization error, similar to validation using unseen data.
